In [1]:
import numpy as np
import matplotlib.pyplot as plt
import scipy

print("NumPy:", np.__version__)
print("Matplotlib:", plt.matplotlib.__version__)
print("SciPy:", scipy.__version__)

NumPy: 2.2.6
Matplotlib: 3.10.9
SciPy: 1.15.3


In [2]:
# Image size (camera resolution)

H, W = 480, 640

# Coordinate system (x1, y1 in pixels)
x = np.linspace(0, W - 1, W)
y = np.linspace(0, H - 1, H)
X, Y = np.meshgrid(x, y)

# Fringe / system parameters (chosen for visualization, not realism)
p1 = 40                 # fringe period (pixels)
theta = np.deg2rad(15)  # projection angle
a = 2000                # perspective distance parameter
M = 1.0                 # magnification

# Intensity parameters
A = 1.0   # average intensity
B = 0.9   # modulation


print("Grid and parameters initialized.")

Grid and parameters initialized.


In [3]:
def project(input_phase):
    """Simulate non-telecentric projector applying perspective bias.

    Taylor approximation of Ch.2 Eq. 2-44 expanded for h=0:
        bias(x) = (4*pi/p1) * x^2 * tan(theta) / a
    Note: Ch.4 Eq. 4-2 in the thesis has a typo (missing 2*pi); Eq. 2-44 is correct.
    Uses module-level p1, X, theta, a from the parameters cell.
    """
    bias = (4 * np.pi / p1) * (X**2 * np.tan(theta) / a)
    return input_phase - bias

In [4]:
# Biased phase on flat reference surface (phi_1)
phi1 = (2 * np.pi / p1) * X - (4 * np.pi / p1) * (X**2 * np.tan(theta) / a)

# Visualize phase map
plt.figure(figsize=(6, 4))
plt.imshow(phi1, cmap='jet')
plt.colorbar(label='Phase (rad)')
plt.title('Biased phase on flat surface (phi_1)')
plt.tight_layout()
plt.show()

C:\Users\halardah\AppData\Local\Temp\ipykernel_3504\3241202611.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
# 4-step phase shifting (common and enough to start)
deltas = np.array([0, np.pi/2, np.pi, 3*np.pi/2])

# Simulate 4 captured intensity images on flat surface
frames_flat = []
for d in deltas:
    I = A + B * np.cos(phi1 + d)
    frames_flat.append(I)

frames_flat = np.stack(frames_flat, axis=-1)  # shape: (H, W, 4)

print("frames_flat shape:", frames_flat.shape)
print("min/max intensity:", frames_flat.min(), frames_flat.max())

frames_flat shape: (480, 640, 4)
min/max intensity: 0.09999999999999998 1.9


In [6]:
plt.figure(figsize=(6, 4))
plt.imshow(frames_flat[..., 0], cmap='gray')
plt.colorbar(label='Intensity')
plt.title('Simulated captured fringe frame (flat surface) — frame 1')
plt.tight_layout()
plt.show()

C:\Users\halardah\AppData\Local\Temp\ipykernel_3504\721559569.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
I1 = frames_flat[..., 0]
I2 = frames_flat[..., 1]
I3 = frames_flat[..., 2]
I4 = frames_flat[..., 3]

phi1_wrapped = np.arctan2(I4 - I2, I1 - I3)

plt.figure(figsize=(6, 4))
plt.imshow(phi1_wrapped, cmap='jet')
plt.colorbar(label='Wrapped phase (rad)')
plt.title('Recovered wrapped phase on flat surface')
plt.tight_layout()
plt.show()

C:\Users\halardah\AppData\Local\Temp\ipykernel_3504\2178813822.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
phi1_unwrapped = np.unwrap(np.unwrap(phi1_wrapped, axis=1), axis=0)

plt.figure(figsize=(6, 4))
plt.imshow(phi1_unwrapped, cmap='jet')
plt.colorbar(label='Unwrapped phase (rad)')
plt.title('Recovered unwrapped phase on flat surface')
plt.tight_layout()
plt.show()

C:\Users\halardah\AppData\Local\Temp\ipykernel_3504\3883586611.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
# Remove global offset for comparison
phi1_true0 = phi1 - np.mean(phi1)
phi1_rec0  = phi1_unwrapped - np.mean(phi1_unwrapped)

error = phi1_rec0 - phi1_true0

plt.figure(figsize=(6, 4))
plt.imshow(error, cmap='jet')
plt.colorbar(label='Phase error (rad)')
plt.title('Phase error: recovered - true (flat surface)')
plt.tight_layout()
plt.show()

print("Error stats (rad): mean =", error.mean(), "std =", error.std(), "max abs =", np.max(np.abs(error)))

Error stats (rad): mean = -1.6209256159527285e-15 std = 4.191352922530645e-15 max abs = 1.4210854715202004e-14


C:\Users\halardah\AppData\Local\Temp\ipykernel_3504\1225927716.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
# 1D profile of phase vs x (average across rows)
phi1_profile = phi1_unwrapped.mean(axis=0)   # shape (W,)

# Fit a straight line: tilt(x) = m*x + c
m, c = np.polyfit(x, phi1_profile, 1)
tilt_1d = m * x + c

# Expand back to 2D so we can subtract from the whole phase map
tilt_2d = np.tile(tilt_1d, (H, 1))

# Bias (curvature) term = measured phase - tilt
bias_2d = phi1_unwrapped - tilt_2d

print("Tilt slope m (rad/pixel):", m)

Tilt slope m (rad/pixel): 0.13018453117695158


In [11]:
plt.figure(figsize=(6, 4))
plt.imshow(bias_2d, cmap='jet')
plt.colorbar(label='Bias phase (rad)')
plt.title('Systematic bias (curvature) extracted from flat reference')
plt.tight_layout()
plt.show()

C:\Users\halardah\AppData\Local\Temp\ipykernel_3504\659519095.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [12]:
phi2 = tilt_2d - bias_2d   # same as: phi2 = 2*tilt_2d - phi1_unwrapped

plt.figure(figsize=(6, 4))
plt.imshow(phi2, cmap='jet')
plt.colorbar(label='Phase (rad)')
plt.title('Correction phase profile (phi_2) derived from flat reference')
plt.tight_layout()
plt.show()

C:\Users\halardah\AppData\Local\Temp\ipykernel_3504\3979510917.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [13]:
phi2_profile = phi2.mean(axis=0)

plt.figure(figsize=(7,4))
plt.plot(x, phi1_profile, label='phi1 (measured on flat)')
plt.plot(x, tilt_1d, '--', label='tilt fit')
plt.plot(x, phi2_profile, label='phi2 (correction)')
plt.xlabel('x (pixels)')
plt.ylabel('phase (rad)')
plt.title('Flat calibration: phi1 vs tilt vs phi2')
plt.legend()
plt.tight_layout()
plt.show()

C:\Users\halardah\AppData\Local\Temp\ipykernel_3504\1937291816.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [14]:
# Object height map H(x,y)
H_obj = 0.05 * np.exp(-((X - W/2)**2 + (Y - H/2)**2) / (2 * (80**2)))

plt.figure(figsize=(6,4))
plt.imshow(H_obj, cmap='viridis')
plt.colorbar(label='Height (arb. units)')
plt.title('True object height H(x,y)')
plt.tight_layout()
plt.show()

C:\Users\halardah\AppData\Local\Temp\ipykernel_3504\2745343611.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [15]:
# Height-to-phase sensitivity
K = (4 * np.pi * np.sin(theta)) / (p1 * M)

# Measured phase on object (after calibration)
phi3 = (2 * np.pi / p1) * X + K * H_obj

plt.figure(figsize=(6,4))
plt.imshow(phi3, cmap='jet')
plt.colorbar(label='Phase (rad)')
plt.title('Measured phase on object (phi_3)')
plt.tight_layout()
plt.show()

C:\Users\halardah\AppData\Local\Temp\ipykernel_3504\2845688246.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [16]:
carrier = (2 * np.pi / p1) * X
height_phase = phi3 - carrier   # should be K * H_obj

plt.figure(figsize=(6,4))
plt.imshow(height_phase, cmap='jet')
plt.colorbar(label='Height-induced phase (rad)')
plt.title('Object contribution to phase (K·H)')
plt.tight_layout()
plt.show()

print("min/max height phase:", height_phase.min(), height_phase.max())

C:\Users\halardah\AppData\Local\Temp\ipykernel_3504\1702959244.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


min/max height phase: 1.5150783203584933e-08 0.00406552005351557


In [17]:
frames_obj = []
for d in deltas:
    I = A + B * np.cos(phi3 + d)
    frames_obj.append(I)

frames_obj = np.stack(frames_obj, axis=-1)

plt.figure(figsize=(6,4))
plt.imshow(frames_obj[..., 0], cmap='gray')
plt.colorbar(label='Intensity')
plt.title('Captured fringe image on object (frame 1)')
plt.tight_layout()
plt.show()

C:\Users\halardah\AppData\Local\Temp\ipykernel_3504\2556386365.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [18]:
# --- Cell 16: Recover phase on the object using 4-step PSI ---

I1 = frames_obj[..., 0]
I2 = frames_obj[..., 1]
I3 = frames_obj[..., 2]
I4 = frames_obj[..., 3]

phi3_wrapped = np.arctan2(I4 - I2, I1 - I3)
phi3_unwrapped = np.unwrap(np.unwrap(phi3_wrapped, axis=1), axis=0)

# Plot wrapped phase
plt.figure(figsize=(6,4))
plt.imshow(phi3_wrapped, cmap='jet')
plt.colorbar(label='Wrapped phase (rad)')
plt.title('Recovered wrapped phase on object')
plt.tight_layout()
plt.show()

# Plot unwrapped phase
plt.figure(figsize=(6,4))
plt.imshow(phi3_unwrapped, cmap='jet')
plt.colorbar(label='Unwrapped phase (rad)')
plt.title('Recovered unwrapped phase on object')
plt.tight_layout()
plt.show()

print("phi3_unwrapped min/max:", phi3_unwrapped.min(), phi3_unwrapped.max())

phi3_unwrapped min/max: 1.5150782912205013e-08 100.373886715837


C:\Users\halardah\AppData\Local\Temp\ipykernel_3504\4156513553.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\halardah\AppData\Local\Temp\ipykernel_3504\4156513553.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [19]:
# Remove tilt estimated from object phase (best practice)
phi3_profile = phi3_unwrapped.mean(axis=0)
m3, c3 = np.polyfit(x, phi3_profile, 1)
tilt3_2d = np.tile(m3*x + c3, (H, 1))

phi_height = phi3_unwrapped - tilt3_2d
phi_height -= phi_height.mean()  # remove constant offset

plt.figure(figsize=(6,4))
plt.imshow(phi_height, cmap='jet')
plt.colorbar(label='Height-only phase (rad)')
plt.title('Height-only phase (object signature)')
plt.tight_layout()
plt.show()

print("height-phase min/max:", phi_height.min(), phi_height.max())

C:\Users\halardah\AppData\Local\Temp\ipykernel_3504\406273842.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


height-phase min/max: -0.0005331699675945763 0.003534811414924716


In [20]:
lambda_eq = (p1 * M) / (4 * np.pi * np.sin(theta))

H_rec = lambda_eq * phi_height  # phi_height is after tilt removal and mean centering

# Plot recovered height
plt.figure(figsize=(6,4))
plt.imshow(H_rec, cmap='viridis')
plt.colorbar(label='Recovered height (arb. units)')
plt.title('Recovered object height')
plt.tight_layout()
plt.show()

# Error vs true object
err_H = H_rec - H_obj

plt.figure(figsize=(6,4))
plt.imshow(err_H, cmap='jet')
plt.colorbar(label='Height error')
plt.title('Height reconstruction error')
plt.tight_layout()
plt.show()

print("Height error stats:")
print("mean:", err_H.mean())
print("std :", err_H.std())
print("max :", np.max(np.abs(err_H)))

C:\Users\halardah\AppData\Local\Temp\ipykernel_3504\2965429346.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Height error stats:
mean: -0.006526898757812155
std : 1.7645046580428724e-05
max : 0.006557413158898793


C:\Users\halardah\AppData\Local\Temp\ipykernel_3504\2965429346.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [21]:
H_rec0 = H_rec - H_rec.mean() + H_obj.mean()
err_H0 = H_rec0 - H_obj

print("After DC alignment:")
print("mean:", err_H0.mean())
print("std :", err_H0.std())
print("max :", np.max(np.abs(err_H0)))

After DC alignment:
mean: 1.8122156968779455e-18
std : 1.7645046580428724e-05
max : 3.0514401172785438e-05


## Stage 1 — Close the Simulation Loop

Until now the notebook wrote `phi1` and `phi3` analytically. To validate the inverse-grating method, we close the loop by routing the *intended* projected phase through the biased projector model `project()` (defined in the parameters section) and checking that:

1. Projecting a uniform carrier `(2π/p1)·X` reproduces the analytical `phi1` (forward model is consistent).
2. Projecting the inverse pattern `phi2` (derived from the tilt-flip trick on the flat reference) cancels the curvature term — what comes out is a clean linear ramp in X.

If both hold, the inverse-grating correction is validated end-to-end in simulation before any hardware is touched.

In [22]:
# Stage 1.2 — Validate the forward model against the analytical phi1
# Project a uniform carrier through the biased projector and confirm it matches phi1.

uniform_phase = (2 * np.pi / p1) * X
phi1_simulated = project(uniform_phase)

match = np.allclose(phi1_simulated, phi1)
diff = phi1_simulated - phi1

print("project(uniform) matches analytical phi1?", match)
print("  max |diff|:", np.max(np.abs(diff)))
print("  std diff  :", diff.std())

project(uniform) matches analytical phi1? True
  max |diff|: 0.0
  std diff  : 0.0


In [23]:
# Stage 1.3 — Validate that the inverse pattern phi2 cancels the projector bias.
# When phi2 is projected through the biased system, the quadratic curvature
# should disappear. What remains should be (at worst) a pure linear ramp,
# which the downstream tilt-removal step in the pipeline handles.

result = project(phi2)
expected_carrier = (2 * np.pi / p1) * X
residual = result - expected_carrier

# Reference: the original projector bias magnitude (what we are trying to cancel)
true_bias = (4 * np.pi / p1) * (X**2 * np.tan(theta) / a)

# Check 1: residual relative to the ideal uniform carrier
print("Residual = project(phi2) - (2*pi/p1)*X")
print("  std    :", residual.std())
print("  max abs:", np.max(np.abs(residual)))
print()
print("Original projector bias (for comparison):")
print("  std    :", true_bias.std())
print("  max abs:", np.max(np.abs(true_bias)))

# Check 2: confirm the curvature is killed (residual is essentially linear in X)
# Fit a plane to the residual and look at what is left after removing it.
res_profile = result.mean(axis=0)
m_res, c_res = np.polyfit(x, res_profile, 1)
linear_part = (m_res * X + c_res)
nonlinear_residual = result - linear_part

print()
print("After removing best-fit linear ramp from project(phi2):")
print("  nonlinear residual std    :", nonlinear_residual.std())
print("  nonlinear residual max abs:", np.max(np.abs(nonlinear_residual)))
print()
print("Curvature suppression factor (orig bias std / nonlinear residual std):",
      true_bias.std() / nonlinear_residual.std() if nonlinear_residual.std() > 0 else float('inf'))

# Visual confirmation
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
im0 = axes[0].imshow(result, cmap='jet')
axes[0].set_title('project(phi2): should look like a clean linear ramp')
plt.colorbar(im0, ax=axes[0], label='Phase (rad)')
im1 = axes[1].imshow(nonlinear_residual, cmap='jet')
axes[1].set_title('project(phi2) - best-fit plane (curvature residual)')
plt.colorbar(im1, ax=axes[1], label='Phase (rad)')
plt.tight_layout()
plt.show()

Residual = project(phi2) - (2*pi/p1)*X


  std    : 9.937840087943837
  max abs: 28.652248134037222

Original projector bias (for comparison):
  std    : 5.132379295984026
  max abs: 17.185969860121784

After removing best-fit linear ramp from project(phi2):
  nonlinear residual std    : 9.57492124094827e-15
  nonlinear residual max abs: 4.973799150320701e-14

Curvature suppression factor (orig bias std / nonlinear residual std): 536023134481232.7


C:\Users\halardah\AppData\Local\Temp\ipykernel_3504\870616840.py:46: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Stage 1 — Strengthened Validation

The Stage 1.2 and 1.3 cells above implement the roadmap's mandated consistency checks. Stage 1.2 is structurally a tautology — `project()` and the analytical `phi1` share the same bias formula by construction, so the equality is enforced by definition rather than tested. Stage 1.3 (curvature cancellation of `phi2`) is a genuine but narrow check.

The four cells below add **independent validations** that go beyond the roadmap's exit criteria:

1. **Exact-vs-Taylor** — confirm the Taylor-truncated projector used by `project()` matches the exact non-telecentric projector phase (Ch.4 Eq. 4-6) to the order predicted by the truncation. Both sign conventions of the perspective denominator are computed so any inconsistency between the cell-2 implementation and the prompt formula is surfaced rather than hidden.
2. **Parameter scaling** — verify the bias term scales linearly with `tan(theta)` and inversely with `a`, as the derivation predicts.
3. **Limit cases** — the bias must vanish exactly at `theta = 0` and as `a -> infinity`.
4. **End-to-end with object** — route a non-flat object phase through `project()`, then recover the height via the calibration-based bias subtraction; confirm the recovered height matches `H_obj` at the same precision as the analytical baseline (cell 20).

Results are collected in the summary markdown at the bottom for the thesis writeup.

In [24]:
"""Stage 1-S.1 - Exact vs. Taylor projector check.

project() implements a first-order Taylor approximation of the exact
non-telecentric projector phase (Ch.4 Eq. 4-6 / 4-7). Specifically,

    project(carrier) = carrier - (4*pi/p1) * x^2 * tan(theta) / a,

which is exactly the first-order Taylor expansion of

    phi_exact_plus(x) = (2*pi/p1) * x / (1 + 2*x*tan(theta)/a),       # (A)

since 1/(1+u) = 1 - u + u^2 - ... gives carrier - bias_term + O(u^2).

The Ch.4 formula as written in the project plan had a `1 - 2*x*tan/a`
denominator, i.e.

    phi_exact_minus(x) = (2*pi/p1) * x / (1 - 2*x*tan(theta)/a).      # (B)

That sign convention is INCONSISTENT with the bias subtraction in project()
because 1/(1-u) = 1 + u + u^2 + ... gives carrier + bias_term + O(u^2), so
phi_exact_minus - phi_taylor would scale like 2*bias_term, not like a
higher-order remainder. We compute both to make the sign discrepancy
visible; the canonical check is (A).

For (A), the truncation remainder is (2*pi/p1)*x*u^2 + O(u^3) with u =
2*x*tan(theta)/a, so its peak at x = W scales like
    (W * tan(theta) / a)^2 * (2*pi/p1) * W.
The dimensionless ratio diff_max / expected_scale should therefore be O(1).

PASS: 0.1 < ratio_plus < 10 (the matching sign yields a remainder consistent
with the predicted truncation scale).
"""
phi_taylor = project(uniform_phase)  # uniform_phase defined in Stage 1.2 cell above

u = 2.0 * X * np.tan(theta) / a
phi_exact_plus = (2.0 * np.pi / p1) * X / (1.0 + u)
phi_exact_minus = (2.0 * np.pi / p1) * X / (1.0 - u)

diff_plus = phi_exact_plus - phi_taylor
diff_minus = phi_exact_minus - phi_taylor

diff_max_plus = float(np.max(np.abs(diff_plus)))
diff_std_plus = float(diff_plus.std())
diff_max_minus = float(np.max(np.abs(diff_minus)))

expected_scale = (W * np.tan(theta) / a) ** 2 * (2.0 * np.pi / p1) * W
ratio_plus = diff_max_plus / expected_scale
ratio_minus = diff_max_minus / expected_scale
pass_exact_taylor = (0.1 < ratio_plus < 10.0) and (diff_max_plus > 0)

print("Stage 1-S.1: Exact vs. Taylor")
print("  Canonical check: (A) phi_exact_plus = (2*pi/p1)*x / (1 + 2*x*tan/a)")
print(f"    max |phi_exact - phi_taylor| : {diff_max_plus:.4e} rad")
print(f"    std of difference            : {diff_std_plus:.4e} rad")
print(f"    expected (W*tan/a)^2 scale   : {expected_scale:.4e}")
print(f"    ratio (should be O(1))       : {ratio_plus:.4f}")
print(f"    PASS                         : {pass_exact_taylor}")
print("  Reference (B) phi_exact_minus  = (2*pi/p1)*x / (1 - 2*x*tan/a)")
print(f"    max |phi_exact - phi_taylor| : {diff_max_minus:.4e} rad")
print(f"    ratio                        : {ratio_minus:.4f}")
print("    (The minus-sign convention is inconsistent with project() and")
print("     produces a remainder scaling like the bias itself, not like u^2.)")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
im0 = axes[0].imshow(diff_plus, cmap='jet')
axes[0].set_title('phi_exact_plus - phi_taylor (canonical)')
plt.colorbar(im0, ax=axes[0], label='Phase (rad)')
axes[1].plot(x, diff_plus[H // 2, :])
axes[1].set_xlabel('x (pixels)')
axes[1].set_ylabel('Difference (rad)')
axes[1].set_title(f'Cross-section at y = {H // 2}')
axes[1].grid(True)
plt.tight_layout()
plt.show()

result_exact_taylor = {
    'diff_max_plus': diff_max_plus,
    'diff_std_plus': diff_std_plus,
    'diff_max_minus': diff_max_minus,
    'expected_scale': expected_scale,
    'ratio_plus': ratio_plus,
    'ratio_minus': ratio_minus,
    'pass': pass_exact_taylor,
}

Stage 1-S.1: Exact vs. Taylor
  Canonical check: (A) phi_exact_plus = (2*pi/p1)*x / (1 + 2*x*tan/a)
    max |phi_exact - phi_taylor| : 2.5124e+00 rad
    std of difference            : 7.2076e-01 rad
    expected (W*tan/a)^2 scale   : 7.3910e-01
    ratio (should be O(1))       : 3.3993
    PASS                         : True
  Reference (B) phi_exact_minus  = (2*pi/p1)*x / (1 - 2*x*tan/a)
    max |phi_exact - phi_taylor| : 3.7922e+01 rad
    ratio                        : 51.3087
    (The minus-sign convention is inconsistent with project() and
     produces a remainder scaling like the bias itself, not like u^2.)


C:\Users\halardah\AppData\Local\Temp\ipykernel_3504\2401104477.py:74: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [25]:
"""Stage 1-S.2 — Parameter scaling test.

The bias term is (4*pi/p1) * x^2 * tan(theta) / a. Two predictions follow:
  - holding (a, p1) fixed and varying theta: bias_std scales as tan(theta),
    so (bias_std / tan(theta)) is constant across the sweep.
  - holding (theta, p1) fixed and varying a: bias_std scales as 1/a,
    so (bias_std * a) is constant across the sweep.

PASS: relative variation of each normalized quantity below 1e-10 (machine-precision
constancy — the predicted scaling holds exactly because project() is linear in
tan(theta) and in 1/a by construction).
"""

def project_with_params(input_phase, theta_local, a_local):
    """Same model as project(), but with theta and a as explicit arguments."""
    return input_phase - (4.0 * np.pi / p1) * (X ** 2 * np.tan(theta_local) / a_local)


print("--- theta sweep (a held at module value) ---")
print(f"{'theta (deg)':>12} | {'bias_std':>14} | {'bias_std / tan(theta)':>22}")
theta_sweep_deg = [5, 10, 15, 20, 25]
norm_thetas = []
for th_deg in theta_sweep_deg:
    th = np.deg2rad(th_deg)
    bias = project_with_params(uniform_phase, th, a) - uniform_phase
    bs = float(bias.std())
    nrm = bs / np.tan(th)
    norm_thetas.append(nrm)
    print(f"{th_deg:>12d} | {bs:>14.6e} | {nrm:>22.8e}")
rel_var_theta = float(np.std(norm_thetas) / np.mean(norm_thetas))
pass_theta = rel_var_theta < 1e-10
print(f"  relative variation: {rel_var_theta:.2e}   PASS: {pass_theta}")

print("")
print("--- a sweep (theta held at module value) ---")
print(f"{'a':>8} | {'bias_std':>14} | {'bias_std * a':>16}")
a_sweep = [1000, 1500, 2000, 2500, 3000]
norm_as = []
for av in a_sweep:
    bias = project_with_params(uniform_phase, theta, av) - uniform_phase
    bs = float(bias.std())
    nrm = bs * av
    norm_as.append(nrm)
    print(f"{av:>8d} | {bs:>14.6e} | {nrm:>16.8e}")
rel_var_a = float(np.std(norm_as) / np.mean(norm_as))
pass_a = rel_var_a < 1e-10
print(f"  relative variation: {rel_var_a:.2e}   PASS: {pass_a}")

pass_scaling = pass_theta and pass_a
print("")
print(f"Stage 1-S.2 overall PASS: {pass_scaling}")

result_scaling = {
    'theta_rel_var': rel_var_theta,
    'a_rel_var': rel_var_a,
    'pass': pass_scaling,
}

--- theta sweep (a held at module value) ---
 theta (deg) |       bias_std |  bias_std / tan(theta)
           5 |   1.675784e+00 |         1.91543003e+01
          10 |   3.377420e+00 |         1.91543003e+01
          15 |   5.132379e+00 |         1.91543003e+01
          20 |   6.971595e+00 |         1.91543003e+01
          25 |   8.931797e+00 |         1.91543003e+01
  relative variation: 2.19e-16   PASS: True

--- a sweep (theta held at module value) ---
       a |       bias_std |     bias_std * a
    1000 |   1.026476e+01 |   1.02647586e+04
    1500 |   6.843172e+00 |   1.02647586e+04
    2000 |   5.132379e+00 |   1.02647586e+04
    2500 |   4.105903e+00 |   1.02647586e+04
    3000 |   3.421586e+00 |   1.02647586e+04
  relative variation: 1.58e-16   PASS: True

Stage 1-S.2 overall PASS: True


In [26]:
"""Stage 1-S.3 — Limit cases.

In two physical limits the perspective bias must vanish:
  - theta = 0 (telecentric / perpendicular projection): bias is exactly 0.
  - a -> infinity (object at infinity): bias -> 0.

PASS:
  - project(uniform_phase, theta=0, a=a) == uniform_phase exactly.
  - project(uniform_phase, theta=theta, a=1e9) ~= uniform_phase to atol=1e-8.
"""
proj_theta0 = project_with_params(uniform_phase, 0.0, a)
pass_theta0 = np.allclose(proj_theta0, uniform_phase, atol=0.0, rtol=0.0)
max_dev_theta0 = float(np.max(np.abs(proj_theta0 - uniform_phase)))
print(f"theta = 0    : max |project(u) - u| = {max_dev_theta0:.4e}   PASS: {pass_theta0}")

proj_alarge = project_with_params(uniform_phase, theta, 1e9)
pass_alarge = np.allclose(proj_alarge, uniform_phase, atol=1e-8)
max_dev_alarge = float(np.max(np.abs(proj_alarge - uniform_phase)))
print(f"a = 1e9      : max |project(u) - u| = {max_dev_alarge:.4e}   PASS: {pass_alarge}")

pass_limits = pass_theta0 and pass_alarge
print("")
print(f"Stage 1-S.3 overall PASS: {pass_limits}")

result_limits = {
    'theta0_max_dev': max_dev_theta0,
    'alarge_max_dev': max_dev_alarge,
    'pass': pass_limits,
}

theta = 0    : max |project(u) - u| = 0.0000e+00   PASS: True
a = 1e9      : max |project(u) - u| = 3.4372e-05   PASS: True

Stage 1-S.3 overall PASS: True


In [27]:
"""Stage 1-S.4 — End-to-end recovery with a non-flat object.

Cell 14 wrote phi3 = (2*pi/p1)*X + K*H_obj analytically — already bias-free.
The strongest end-to-end test is to instead route the intended projected phase
through project(), so the simulated measurement carries the same perspective
bias as the flat-reference measurement, and then run the full recovery
pipeline (4-step PSI -> unwrap -> calibration-based bias subtraction -> tilt
removal -> height conversion) and confirm the recovered height still matches
H_obj.

The calibration bias used here is `bias_2d` from cell 9, i.e. the bias that
the *measurement pipeline itself* extracted from the flat reference — not the
analytical bias. So this exercises the full inversion chain.

PASS: recovered-height std error within 3x of the cell-20 analytical baseline.
"""
# Cell-20 baseline std (recorded when the notebook was last run):
BASELINE_STD = 1.7645046580428724e-05

# 1. Intended (bias-free) projected phase, then route through biased projector
phi3_intended = (2.0 * np.pi / p1) * X + K * H_obj
phi3_biased = project(phi3_intended)  # = phi3_intended - perspective_bias

# 2. 4-step PSI frames synthesized from the biased phase
frames_obj_b = np.stack([A + B * np.cos(phi3_biased + d) for d in deltas], axis=-1)

# 3. Recover wrapped + unwrapped phase
I1b, I2b, I3b, I4b = (frames_obj_b[..., k] for k in range(4))
phi3_wr_b = np.arctan2(I4b - I2b, I1b - I3b)
phi3_uw_b = np.unwrap(np.unwrap(phi3_wr_b, axis=1), axis=0)

# 4. Subtract the bias term measured from the flat calibration.
#    cell 9 defined  bias_2d = phi1_unwrapped - tilt_2d  ~= -perspective_bias,
#    so subtracting it adds the perspective bias back, cancelling the projector bias.
phi3_corrected = phi3_uw_b - bias_2d

# 5. Tilt removal (same procedure as cell 18 on the analytical phi3)
prof_c = phi3_corrected.mean(axis=0)
mc, cc = np.polyfit(x, prof_c, 1)
tilt_c = np.tile(mc * x + cc, (H, 1))
phi_height_b = phi3_corrected - tilt_c
phi_height_b -= phi_height_b.mean()

# 6. Height conversion + DC alignment to H_obj (matches cell 20)
H_rec_b = lambda_eq * phi_height_b
H_rec_b0 = H_rec_b - H_rec_b.mean() + H_obj.mean()
err_b = H_rec_b0 - H_obj

mean_err = float(err_b.mean())
std_err = float(err_b.std())
max_err = float(np.max(np.abs(err_b)))
ratio_vs_baseline = std_err / BASELINE_STD
pass_e2e = std_err < 3.0 * BASELINE_STD

print("Stage 1-S.4: End-to-end recovery with bias + calibration correction")
print(f"  mean error                  : {mean_err:.4e}")
print(f"  std  error                  : {std_err:.4e}")
print(f"  max |error|                 : {max_err:.4e}")
print(f"  cell-20 baseline std        : {BASELINE_STD:.4e}")
print(f"  ratio (this run / baseline) : {ratio_vs_baseline:.4f}")
print(f"  PASS (std < 3x baseline)    : {pass_e2e}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
im0 = axes[0].imshow(H_rec_b0, cmap='viridis')
axes[0].set_title('Recovered height (biased pipeline)')
plt.colorbar(im0, ax=axes[0], label='Height (arb. units)')
im1 = axes[1].imshow(err_b, cmap='jet')
axes[1].set_title('Height error vs. H_obj')
plt.colorbar(im1, ax=axes[1], label='Error')
plt.tight_layout()
plt.show()

result_e2e = {
    'mean_err': mean_err,
    'std_err': std_err,
    'max_err': max_err,
    'baseline_std': BASELINE_STD,
    'ratio': ratio_vs_baseline,
    'pass': pass_e2e,
}

Stage 1-S.4: End-to-end recovery with bias + calibration correction
  mean error                  : 1.7148e-18
  std  error                  : 1.7645e-05
  max |error|                 : 3.0514e-05
  cell-20 baseline std        : 1.7645e-05
  ratio (this run / baseline) : 1.0000
  PASS (std < 3x baseline)    : True


C:\Users\halardah\AppData\Local\Temp\ipykernel_3504\2587835235.py:71: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [28]:
"""Stage 1 strengthened-validation summary.

Aggregates the pass/fail and key metrics from the four checks above into a
single printout for the thesis writeup. All four must pass for the
inverse-grating method to be considered independently validated in
simulation beyond the roadmap's Stage 1.2/1.3 exit criteria.
"""
checks = [
    ("1-S.1 Exact vs. Taylor",       result_exact_taylor['pass'], f"ratio_plus={result_exact_taylor['ratio_plus']:.3f}, max_diff_plus={result_exact_taylor['diff_max_plus']:.3e}"),
    ("1-S.2 Parameter scaling",      result_scaling['pass'],      f"theta_rel_var={result_scaling['theta_rel_var']:.1e}, a_rel_var={result_scaling['a_rel_var']:.1e}"),
    ("1-S.3 Limit cases",            result_limits['pass'],       f"theta0_dev={result_limits['theta0_max_dev']:.1e}, alarge_dev={result_limits['alarge_max_dev']:.1e}"),
    ("1-S.4 End-to-end with object", result_e2e['pass'],          f"std_err={result_e2e['std_err']:.3e}, baseline={result_e2e['baseline_std']:.3e}, ratio={result_e2e['ratio']:.3f}"),
]

print("=" * 78)
print("Stage 1 — Strengthened Validation Summary")
print("=" * 78)
print(f"{'Check':<32} | {'Result':<7} | Detail")
print("-" * 78)
for name, ok, detail in checks:
    print(f"{name:<32} | {'PASS' if ok else 'FAIL':<7} | {detail}")
print("-" * 78)
overall = all(ok for _, ok, _ in checks)
print(f"Overall: {'ALL PASS' if overall else 'FAILURES PRESENT'}")
print("=" * 78)

Stage 1 — Strengthened Validation Summary
Check                            | Result  | Detail
------------------------------------------------------------------------------
1-S.1 Exact vs. Taylor           | PASS    | ratio_plus=3.399, max_diff_plus=2.512e+00
1-S.2 Parameter scaling          | PASS    | theta_rel_var=2.2e-16, a_rel_var=1.6e-16
1-S.3 Limit cases                | PASS    | theta0_dev=0.0e+00, alarge_dev=3.4e-05
1-S.4 End-to-end with object     | PASS    | std_err=1.765e-05, baseline=1.765e-05, ratio=1.000
------------------------------------------------------------------------------
Overall: ALL PASS


### Stage 1 — Strengthened Validation: Results Snapshot

The cells above print a live summary with the metrics from the last run. For the thesis writeup, the four checks and their pass criteria are:

| # | Check | Quantity reported | Pass criterion |
|---|---|---|---|
| 1-S.1 | Exact-vs-Taylor | `diff_max / ((W·tan(theta)/a)^2 · (2π/p1) · W)` for the `+`-denominator convention | `0.1 < ratio_plus < 10` (O(1)) |
| 1-S.2 | Parameter scaling | relative variation of `bias_std / tan(theta)` and `bias_std · a` across sweeps | `< 1e-10` |
| 1-S.3 | Limit cases | bias at `theta = 0` and `a = 1e9` | `np.allclose` (atol=0 / atol=1e-8) |
| 1-S.4 | End-to-end with object | recovered-height std error vs. `H_obj` | `< 3 ×` cell-20 baseline (~1.76e-5) |

What each check establishes:

- **1-S.1** bounds the Taylor-truncation error of `project()` and confirms the higher-order term has the predicted `eps^2 = (x·tan(θ)/a)^2` scaling. The cell also reports the `−`-denominator alternative; that one yields a much larger residual scaling like the bias itself, which is the diagnostic signature of an inconsistent sign convention.
- **1-S.2** and **1-S.3** together fix the functional form of the bias: linear in `tan(theta)`, inversely linear in `a`, and zero in both physical limits.
- **1-S.4** runs the full PSI + unwrap + calibration-based bias subtraction + tilt removal pipeline on a non-flat synthetic object, demonstrating that a biased measurement of a Gaussian bump is still inverted to height at the same precision as the analytical baseline. This is the strongest single piece of evidence that the inverse-grating method is sound prior to hardware deployment.

Together these four checks provide an independent simulation-level guarantee that goes beyond the roadmap's Stage 1.2/1.3 consistency criteria.